# Attribute-gradient measurements

Load a completed diagnostic run. No models are loaded or training launched. Keep streams, training seeds, and prompt blocks separate. See [the run guide](../experiments/run_attribute_diagnostics.md).


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
RUN_DIR = ROOT / "results/conciseness_gradient_collapse/gpt2-alpha075-seed42-pilot"
from src.imdb.diagnostics.analyze_attribute_gradients import analyze
tables = analyze(RUN_DIR)


## Feature table
Each row carries run, seed, alpha, PPO update, stream, and phase. Select `block == "all"` for full-panel gradients. Prompt blocks measure sampling stability, not training-seed replication.


In [ ]:
import pandas as pd
from IPython.display import display, Image
rollouts = pd.DataFrame(tables["rollout_summary"])
gradients = pd.DataFrame(tables["gradient_summary"])
updates = pd.DataFrame(tables["update_summary"])
feature_columns = ["ppo_update", "stream", "phase", "policy_length_mean", "policy_length_std", "cap_hit_rate", "conciseness_std", "sentiment_std"]
display(rollouts.reindex(columns=feature_columns))
display(gradients.loc[gradients["block"].eq("all")].reindex(columns=["ppo_update", "stream", "weighted_norm_conciseness", "weighted_norm_sentiment", "rho", "mixture_error_relative"]))


## Mechanism figures
Positive mixed alignment alone is insufficient. Inspect weighted norms, signed contributions, and actual attribute outcomes together. Null cosines indicate unresolved numerical directions.


In [ ]:
for path in sorted((RUN_DIR / "figures").glob("*.png")):
    print(path.name)
    display(Image(filename=str(path)))


## Actual optimizer contributions
These are first-order estimates for the first synchronized optimizer step at selected rollouts. Empty output is expected for offline checkpoints: no actual update was observed.


In [ ]:
display(updates.reindex(columns=["ppo_update", "learning_rate", "norm_update", "cos_update__minority_rm", "cos_update__minority_oracle", "contribution_conciseness", "contribution_sentiment"]))
display(pd.DataFrame(tables["counterfactual_alpha"]))


## Record findings after inspecting results

- Did length variance fall, or only scored variance?
- Did weighted conciseness norm fall below weighted sentiment norm consistently across blocks and seeds?
- Did the learned minority gradient follow the oracle mixture?
- Were positive update contributions sentiment-driven while conciseness stayed poor?
- Which alternative explanations remain consistent with these measurements?

Do not fill conclusions before measurements exist.
